# Kenya Malaria Analysis (2015–2025)

**Project type:** Data Analytics  
**Study area:** Kenya  
**Study period:** 2015–2025

## Project overview
This notebook explores malaria incidence patterns across Kenya using the supplied county-level dataset. It addresses four research questions:

1. How has malaria incidence changed in Kenya from 2015 to 2025?
2. Which months recorded the highest and lowest average malaria incidence?
3. Which counties recorded the highest and lowest average malaria incidence?
4. Is there a relationship between rainfall and malaria incidence?

The main outcome variable is `incidence_per_1000`. Rainfall is represented by `precipitation`.

**Important:** This is a descriptive analysis. A correlation can show an association, but it cannot prove that rainfall causes malaria.

## 1. Import libraries

We use pandas for data handling, NumPy for numerical work, and Matplotlib for visualisations.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 2. Load the dataset

Place `malaria_kenya.xlsx` in the same folder as this notebook before running this cell.

If your file has a different name or location, update `file_path`.

In [ ]:
file_path = "malaria_kenya.xlsx"
df = pd.read_excel (C:\Users\Admin\Desktop\Gen-data-project\malaria_kenya.xlsx)

print("Dataset shape:", df.shape)
df.head()

SyntaxError: invalid syntax (1897805553.py, line 2)

## 3. Understand and inspect the data

Check the column names, data types, missing values, duplicates, and the date range.

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\\nData types:")
display(df.dtypes)

print("\\nMissing values:")
display(df.isna().sum())

print("\\nDuplicate rows:", df.duplicated().sum())

if "year" in df.columns:
    print("Year range:", df["year"].min(), "to", df["year"].max())

if "county" in df.columns:
    print("Number of counties:", df["county"].nunique())

## 4. Prepare analysis fields

The dataset includes a month field. We create a consistent month number and month name for chronological grouping. The original data is retained in `df`; the prepared data is stored in `data`.

In [ ]:
data = df.copy()

# Ensure key numeric fields are numeric
for column in ["year", "month", "incidence_per_1000", "precipitation"]:
    if column in data.columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

# Create month number/name where possible
if "month" in data.columns:
    data["month_num"] = data["month"].astype("Int64")
    data["month_name"] = data["month_num"].map(
        lambda m: pd.Timestamp(2000, int(m), 1).strftime("%B") if pd.notna(m) and 1 <= int(m) <= 12 else np.nan
    )

required = ["county", "year", "month", "incidence_per_1000", "precipitation"]
missing_columns = [c for c in required if c not in data.columns]
if missing_columns:
    print("Warning: expected columns not found:", missing_columns)

data[["county", "year", "month", "incidence_per_1000", "precipitation"]].head()

## 5. Research Question 1: How has malaria incidence changed from 2015 to 2025?

Calculate the mean incidence for each year and visualise the trend.

In [ ]:
annual_avg = (
    data.groupby("year", as_index=False)["incidence_per_1000"]
        .mean()
        .sort_values("year")
)

display(annual_avg)

plt.figure(figsize=(10, 5))
plt.plot(annual_avg["year"], annual_avg["incidence_per_1000"], marker="o")
plt.title("Average Malaria Incidence in Kenya (2015–2025)")
plt.xlabel("Year")
plt.ylabel("Average incidence per 1,000 people")
plt.grid(True, alpha=0.3)
plt.xticks(annual_avg["year"], rotation=45)
plt.tight_layout()
plt.show()

highest_year = annual_avg.loc[annual_avg["incidence_per_1000"].idxmax()]
lowest_year = annual_avg.loc[annual_avg["incidence_per_1000"].idxmin()]
change_pct = (lowest_year["incidence_per_1000"] - highest_year["incidence_per_1000"]) / highest_year["incidence_per_1000"] * 100

print(f"Highest annual average: {int(highest_year['year'])} ({highest_year['incidence_per_1000']:.2f})")
print(f"Lowest annual average: {int(lowest_year['year'])} ({lowest_year['incidence_per_1000']:.2f})")
print(f"Change from highest to lowest year: {change_pct:.1f}%")

## 6. Research Question 2: Which months recorded the highest and lowest average incidence?

The monthly averages pool observations from all years. Months are sorted in calendar order.

In [ ]:
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

monthly_avg = (
    data.groupby("month_name")["incidence_per_1000"]
        .mean()
        .reindex(month_order)
        .dropna()
)

display(monthly_avg.rename("average_incidence_per_1000").to_frame())

plt.figure(figsize=(11, 5))
plt.bar(monthly_avg.index, monthly_avg.values)
plt.title("Average Malaria Incidence by Month")
plt.xlabel("Month")
plt.ylabel("Average incidence per 1,000 people")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"Highest month: {monthly_avg.idxmax()} ({monthly_avg.max():.2f})")
print(f"Lowest month: {monthly_avg.idxmin()} ({monthly_avg.min():.2f})")

## 7. Research Question 3: Which counties recorded the highest and lowest average incidence?

Calculate the average incidence for each county and compare the highest and lowest counties.

In [ ]:
county_avg = (
    data.groupby("county")["incidence_per_1000"]
        .mean()
        .sort_values(ascending=False)
)

display(county_avg.to_frame("average_incidence_per_1000"))

top_counties = county_avg.head(10).sort_values()
bottom_counties = county_avg.tail(10)

plt.figure(figsize=(10, 6))
plt.barh(top_counties.index, top_counties.values)
plt.title("10 Counties with the Highest Average Malaria Incidence")
plt.xlabel("Average incidence per 1,000 people")
plt.tight_layout()
plt.show()

print(f"Highest county: {county_avg.idxmax()} ({county_avg.max():.2f})")
print(f"Lowest county: {county_avg.idxmin()} ({county_avg.min():.2f})")

## 8. Research Question 4: Is there a relationship between rainfall and malaria incidence?

Use Pearson correlation to measure the linear association between precipitation and malaria incidence. A scatter plot helps visualise the relationship.

In [ ]:
rain_data = data[["precipitation", "incidence_per_1000"]].dropna()

pearson_r = rain_data["precipitation"].corr(rain_data["incidence_per_1000"], method="pearson")
print(f"Pearson correlation (r): {pearson_r:.3f}")

plt.figure(figsize=(8, 6))
plt.scatter(
    rain_data["precipitation"],
    rain_data["incidence_per_1000"],
    alpha=0.35
)
plt.title("Rainfall vs Malaria Incidence")
plt.xlabel("Precipitation")
plt.ylabel("Malaria incidence per 1,000 people")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

print("Interpretation: a positive value indicates that higher rainfall tends to be associated with higher incidence.")
print("Correlation does not establish causation.")

## 9. Summary of findings

This section generates a compact summary from the dataset so the reported results stay connected to the calculations above.

In [ ]:
summary = {
    "Overall average incidence per 1,000": data["incidence_per_1000"].mean(),
    "Highest annual average year": int(highest_year["year"]),
    "Highest annual average": highest_year["incidence_per_1000"],
    "Lowest annual average year": int(lowest_year["year"]),
    "Lowest annual average": lowest_year["incidence_per_1000"],
    "Highest month": monthly_avg.idxmax(),
    "Highest monthly average": monthly_avg.max(),
    "Lowest month": monthly_avg.idxmin(),
    "Lowest monthly average": monthly_avg.min(),
    "Highest county": county_avg.idxmax(),
    "Highest county average": county_avg.max(),
    "Lowest county": county_avg.idxmin(),
    "Lowest county average": county_avg.min(),
    "Rainfall Pearson correlation": pearson_r,
}

pd.Series(summary, name="Result").to_frame()

## 10. Conclusion

Based on the supplied dataset, summarise:

- The direction of the annual malaria incidence trend.
- The months with the highest and lowest average incidence.
- The counties with the highest and lowest average incidence.
- The strength and direction of the rainfall association.

### Limitations
- This is a descriptive analysis of the supplied dataset.
- Group averages can hide variation within counties, months and years.
- Correlation does not prove causation.
- The analysis does not establish why incidence differs across time or counties.
- Findings should be checked against current official health data before being used for operational or policy decisions.

## Reproducibility

To reproduce the analysis:
1. Install the required packages: `pandas`, `numpy`, `matplotlib`, and `openpyxl`.
2. Put `malaria_kenya.xlsx` in the same folder as this notebook.
3. Run the notebook from top to bottom.